# Multi Agent Research Pipeline

A multi agent system with planner, researcher, writer and editor

In [ ]:
# --- Standard library
from datetime import datetime
import re
import json
import ast

# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [ ]:
CLIENT = Client()

In [ ]:
def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a step-by-step research plan for a multi-agent workflow.

    Args:
        topic: Research topic to investigate.
        model: Language model to use.

    Returns:
        List of executable step strings, one per agent action.
    """
    user_prompt = f"""You are a planning agent responsible for organising a research workflow with multiple intelligent agents.

Available agents:
- A research agent who can search the web, Wikipedia, and arXiv.
- A writer agent who can draft research summaries.
- An editor agent who can reflect and revise the drafts.

Write a clear, step-by-step research plan as a valid Python list, where each step is a string.
Each step should be atomic and executable using only the capabilities of the above agents.
Do NOT include tasks like "create CSV", "set up a repo", or "install packages".
Do include real research tasks (search, summarise, draft, revise).
Return ONLY the Python list — no explanation text.
The final step should produce a Markdown document with the complete research report.

Topic: "{topic}"
"""

    messages = [{"role": "user", "content": user_prompt}]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1,
    )

    steps_str = response.choices[0].message.content.strip()
    steps = ast.literal_eval(steps_str)
    return steps

In [ ]:
def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using arXiv, Tavily, and Wikipedia tools.

    Returns the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")
    print(" Research Agent")
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')

    prompt = (
        f"You are a research assistant with access to web search, Wikipedia, and arXiv. "
        f"Today's date is {current_time}. "
        f"Use the available tools to thoroughly research the following task.\n\nTask: {task}"
    )
    messages = [{"role": "user", "content": prompt}]
    tools = [
        research_tools.arxiv_tool_def,
        research_tools.tavily_tool_def,
        research_tools.wikipedia_tool_def,
    ]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_turns=6,
    )

    content = response.choices[0].message.content
    print(" Output:\n", content)

    return (content, messages) if return_messages else content

In [ ]:
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Drafts, expands, or summarises content for a given writing task.
    """
    print("==================================")
    print("️ Writer Agent")
    print("==================================")

    system_prompt = (
        "You are a writing agent specialised in generating well-structured academic and technical content. "
        "Produce clear, accurate, and well-organised prose."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1.0,
    )

    return response.choices[0].message.content

In [ ]:
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Reflects on, critiques, or revises a research draft.
    """
    print("==================================")
    print(" Editor Agent")
    print("==================================")

    system_prompt = (
        "You are an editor agent specialised in reflecting on, critiquing, and improving research drafts. "
        "Provide constructive, specific feedback and produce polished revisions."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )

    return response.choices[0].message.content

In [ ]:
agent_registry = {
 "research_agent": research_agent,
 "editor_agent": editor_agent,
 "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
 """
 Clean the contents of a JSON block that may come wrapped with Markdown backticks.
 """
 raw = raw.strip()

 if raw.startswith("```"):
  raw = re.sub(r"^```(?:json)?\n?", "", raw)

 raw = re.sub(r"\n?```$", "", raw)

 return raw.strip()

In [ ]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

 plan_steps = planner_agent(topic)
 max_steps = 4

 if limit_steps:
 plan_steps = plan_steps[:min(len(plan_steps), max_steps)]

 history = []

 print("==================================")
 print(" Executor Agent")
 print("==================================")

 for i, step in enumerate(plan_steps):

 agent_decision_prompt = f"""
 You are an execution manager for a multi-agent research team.

 Given the following instruction, identify which agent should perform it and extract the clean task.

 Return only a valid JSON object with two keys:
 - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
 - "task": a string with the instruction that the agent should follow

 Only respond with a valid JSON object. Do not include explanations or markdown formatting.

 Instruction: "{step}"
 """
 response = CLIENT.chat.completions.create(
 model=model,
 messages=[{"role": "user", "content": agent_decision_prompt}],
 temperature=0,
 )

 raw_content = response.choices[0].message.content
 cleaned_json = clean_json_block(raw_content)
 agent_info = json.loads(cleaned_json)

 agent_name = agent_info["agent"]
 task = agent_info["task"]

 context = "\n".join([
 f"Step {j+1} executed by {a}:\n{r}"
 for j, (s, a, r) in enumerate(history)
 ])
 enriched_task = f"""
 You are {agent_name}.

 Here is the context of what has been done so far
 {context}

 Your next task is:
 {task}
 """

 print(f"\n️ Executing with agent: `{agent_name}` on task: {task}")

 if agent_name in agent_registry:
 output = agent_registry[agent_name](enriched_task)
 history.append((step, agent_name, output))
 else:
 output = f"️ Unknown agent: {agent_name}"
 history.append((step, agent_name, output))

 print(f" Output:\n{output}")

 return history

In [ ]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")
display(Markdown(md))